# Grokking on Three-Operand Modulo Addition: Dataset Construction, Training Dynamics and Checkpointing

> _Course_: DD2417 Language Engineering (KTH Royal Institute of Technology)  
> _Instructor_: Prof. Johan Boye  
> _Project group members_: Arianna PAONE, Damien BARDINA (Group 23)  
> _Submission date_: 2nd June 2026  

The main purpose of the present notebook is to define the model architecture ("simplistic" two-block Transformer architecture heavily inspired by our previous Assignment 3) which will be trained excessively to predict the outcomes (viewed as a classification problem) on the modulo arithmetic problem $(a_1 + \ldots + a_n)\bmod p$. This rather "easy" task is designed to be a toy problem for observing the _grokking_ (also called _delayed generalisation_) phenomenon as coined by [Power et al. (2022)](https://arxiv.org/pdf/2201.02177) and later reverse-engineered by [Nanda et al. (2023)](https://arxiv.org/pdf/2301.05217), since it is not trivial enough for the model to memorise/overfit the underlying training dataset right away – but also not too hard to ultimately capture the lower-weight generalisation solution. 

Although being easily modifiable to more operands, we will focus here on the problem of $\bmod p = \bmod 37$ (to reduce memory consumption and problem domain complexity) and 3 operands – thereby representing an answering attempt to some of the open-ended questions in the work by [Power et al. (2022)](https://arxiv.org/pdf/2201.02177).

---

## 1) Import of notebook dependencies

Here we import our external project libraries which are pretty canonical. The only exception is the `plotly` library which was our go-to-choice due to its interactive  capabilities (hovering over datapoints, zoom in/out via mouse dragging, ...).

In [1]:
import os
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch execution environment device:', DEVICE)

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

PyTorch execution environment device: cuda


---

## 2) Hyperparameter configuration setup 

In [2]:
# problem hyperparameters
P            = 37               # modulus number 
N_INPUTS     = 3                # number of addition operands
TRAIN_FRAC   = 0.2              # crucial for grokking occurrence 
                                # (not enough -> fails to generalise, too much -> learns generalising algorithm right away)
# model hyperparameters
D_MODEL      = 128              # embedding dimensionality (for simplicity & efficiency shared for Q,K,V)
N_HEADS      = 4                # number of attention heads (number of splits in d_model)
N_LAYERS     = 2                # number of Transfomer blocks (as done in the Attention is All You Need paper)
D_FF         = 4 * D_MODEL      # dimensionality of hidden layer in position-wise MLP in a Transformer block
CAUSAL       = True            # Boolean flag for enabling bidirectional vs. causal masking in attention

# optimisation hyperparameters
LR           = 1e-3             # learning rate
LR_MIN       = 1e-4             # floor value for cosine annealing to improve stability after grokking
WEIGHT_DECAY = 1.0              # crucial for grokking occurrence (insufficient L2 regularisation does not incentivise grokking)
BETAS        = (0.9, 0.98)      # AdamW parameters
N_STEPS      = 30_000           # max number of full batch samples (with replacement)
LOG_EVERY    = 100
SEED         = 29
DTYPE        = torch.float64    # float64 as medication against "slingshot" mechanism, very small losses, "edge-of-stability" problems
CKPT_DIR     = './ckpts_causal' if CAUSAL==True else './ckpts_noncausal'  # current working directory for our .pt files
os.makedirs(CKPT_DIR, exist_ok=True)

---

## 3) Layer components and model architecture classes

Our implementation for the Transformer block is (as already mentioned above) heavily inspired by one of our previous assignments and is the cornerstone for implementing forward pass "hooks" that store the required quantities for the mechanistic interpretability analysis. 

For convenience, we implemented the model architecture such that causal masking can be enabled/disabled easily (autoregressive vs bidirectional) as it represents an interesting comparison playground in this project in terms of examining the positional learning character. In more jargon terms, one could say that we can then compare a _generative_ (_decoder-only_) approach to a rather _discriminative_ (_encoder-only_) approach.

In [3]:
class Attention(nn.Module):
    # class constructor 
    def __init__(self, d_model, n_heads, causal=True):
        super().__init__()
        self.n_heads, self.d_head, self.causal = n_heads, d_model // n_heads, causal
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)    # coalesced projection matrices Q,K,V for computation efficiency
        self.W_o   = nn.Linear(d_model, d_model, bias=False)        # output projection for concatenated multi-head output

    
    # forward pass through attention layer 
    def forward(self, x, return_attn=False):
        B, T, C = x.shape
        
        q, k, v = self.W_qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)  # attention scaling for more stable convergence 
        if self.causal:                                              # optional causal mask
            mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
            scores = scores.masked_fill(mask, float('-inf'))
            
        attn = F.softmax(scores, dim=-1)                             # per-query (row-wise) softmax for attention scores
        contextualised_embedding = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_o(contextualised_embedding)    
        
        return (out, attn) if return_attn else out



class TransformerBlock(nn.Module):
    # class constructor
    def __init__(self, d_model, n_heads, d_ff, causal=True):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(d_model), Attention(d_model, n_heads, causal)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))

    
    # forward pass through Transformer block
    def forward(self, x, return_attn=False):
        a = self.attn(self.ln1(x), return_attn=return_attn)
        if return_attn:
            a, w = a
        
        x = x + a                       # residual connection after LayerNorm + Attention layer
        x = x + self.ffn(self.ln2(x))   # residual connection after LayerNorm + Position-Wise MLP layer
        
        return (x, w) if return_attn else x



class GrokTransformer(nn.Module):
    # class constructor
    def __init__(self, p, n_inputs, d_model, n_heads, n_layers, d_ff, causal=True):
        super().__init__()
        self.p, self.seq_len, self.n_layers = p, n_inputs + 1, n_layers    # +1 for "=" token
        self.token_emb = nn.Embedding(p + 1, d_model)                      # token embedding
        self.pos_emb   = nn.Embedding(self.seq_len, d_model)               # positional embedding
        self.blocks    = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, causal) for _ in range(n_layers)])
        self.ln_f      = nn.LayerNorm(d_model)
        self.unembed   = nn.Linear(d_model, p, bias=False)

    
    # forward pass through entire model
    def forward(self, x, return_cache=False):
        h = self.token_emb(x) + self.pos_emb(torch.arange(x.size(1), device=x.device))

        # faster path used in training (just do foward pass)
        if not return_cache:                              
            for block in self.blocks:
                h = block(h)
            return self.unembed(self.ln_f(h)[:, -1])

        # instrumented path used in analysis (store residual outputs / attention scores after each block)
        cache = {'embed': h}                              
        for i, block in enumerate(self.blocks):
            h, w = block(h, return_attn=True)
            cache[f'resid_post_{i}'] = h
            cache[f'attn_weights_{i}'] = w 
            
        return self.unembed(self.ln_f(h)[:, -1]), cache

---

## 4) Dataset construction 

The dataset construction is pretty straightforward as we simply perform exhaustive combinatorial input of all possible values for the three operands based on the modulo number $p$.

In [4]:
def make_dataset(p, n_inputs):
    # construct grids for all possible input combinations (n_inputs copies)
    grids = torch.meshgrid(*[torch.arange(p)] * n_inputs, indexing='ij')
    
    # stack the n_inputs tensors into tuples and reshape into 2D tensor with all combos
    numbers = torch.stack(grids, dim=-1).reshape(-1, n_inputs)
    
    # "=" token
    eq = torch.full((numbers.size(0), 1), p, dtype=torch.long)
    
    return torch.cat([numbers, eq], dim=-1), numbers.sum(-1) % p    # shape (p^n_inputs, n_inputs + 1)


@torch.no_grad()
def embedding_diagnostics(W):
    ### PCA projection diagnostic 
    # centering of embedding latent dimensions
    Wc = W - W.mean(dim=0, keepdim=True)  
    
    # SVD: U = projection scores on PCA components, S = singular values (variance per component)
    U, S, _ = torch.linalg.svd(Wc, full_matrices=False)  

    # take first two PCA components and then scaling (2D PCA coordinates for each token)
    pca_xy  = (U[:, :2] * S[:2]).float().cpu().numpy()   

    ### Fourier analysis
    # (1) real FFT across tokens (p token embeddings treated as signals of length p), shape (p//2 + 1, d_model)
    # (2) take magnitude of each Fourier component
    # (3) summing across embedding dimensions, shape (p//2 + +1)
    # => spectrum showing how much each frequency is represented in the embedding matrix
    fourier = torch.fft.rfft(W, dim=0).abs().sum(dim=1).float().cpu().numpy() 
    
    return pca_xy, fourier

## 5) Training setup

The following cell follows the "canonical" PyTorch training setup. The only notable change that we implemented was the use of cosine annealing for learning rate scheduling since we wanted to make the ultimately learned Fourier circuit and token embeddings more stable, especially after the grokking phase transition. 

In [5]:
# for reproducibility
torch.manual_seed(SEED)

# construction of train/val dataset splits
inputs, targets = make_dataset(P, N_INPUTS)
n_total = inputs.size(0)
perm = torch.randperm(n_total, generator=torch.Generator().manual_seed(SEED))
n_train = int(n_total * TRAIN_FRAC)
train_idx, val_idx = perm[:n_train], perm[n_train:]

train_x = inputs[train_idx].to(DEVICE); train_y = targets[train_idx].to(DEVICE)
val_x = inputs[val_idx].to(DEVICE); val_y = targets[val_idx].to(DEVICE)

# model, optimiser (AdamW) and scheduler (cosine annealing) instantiations
model = GrokTransformer(P, N_INPUTS, D_MODEL, N_HEADS, N_LAYERS, D_FF, CAUSAL).to(DEVICE).to(DTYPE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=BETAS, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_STEPS, eta_min=LR_MIN)
task_str = ' + '.join('abcdef'[:N_INPUTS])
print(f'Task: ({task_str}) mod {P} | causal={CAUSAL} | dtype={DTYPE}')
print(f'Train: {n_train}   Val: {n_total - n_train}   Params: {sum(p.numel() for p in model.parameters()):,}')

# checkpointing at regular interval every X steps (plus final one at the end, see training loop below)
# -> multiple of LOG_EVERY so that it lands on periodic evaluation during training
CHECKPOINT_EVERY = 1_000
assert CHECKPOINT_EVERY % LOG_EVERY == 0, 'CHECKPOINT_EVERY must be a multiple of LOG_EVERY'

Task: (a + b + c) mod 37 | causal=True | dtype=torch.float64
Train: 10130   Val: 40523   Params: 405,888


## 6) Grokking training loop (with live dashboard to observe dynamics)

This is also where the actual checkpointing takes place which will allow us to perform the mechanistic interpretability analysis at different stages of model training: random initialisation -> memorisation/overfitting -> grokking phase transition -> frequency selection on grokking plateau.

In [6]:
### LIVE DASHBOARD CODE ###
# The following plotly-based dashboard code was implemented via Claude Opus 4.8 since 
# it was deemed non-educative and too time-consuming to dig into the technical documentation
# for visualisation purposes, allowing to make formatting and labelling changes way quicker via
# the use of AI.

# Figure 1: cross-entropy loss curve, accuracy curve, weight norm curve
fig1 = go.FigureWidget(make_subplots(rows=1, cols=3,
    subplot_titles=('Loss', 'Accuracy', 'Weight norm'), horizontal_spacing=0.08))
TC, VC, NC = '#1f77b4', '#d62728', '#9467bd'
for c, col in [(TC,1),(VC,1)]: fig1.add_trace(go.Scatter(x=[],y=[],mode='lines',line=dict(color=c,width=2),
    name=('train' if c==TC else 'val')), 1, col)
fig1.add_trace(go.Scatter(x=[],y=[],mode='lines',line=dict(color=TC,width=2),showlegend=False),1,2)
fig1.add_trace(go.Scatter(x=[],y=[],mode='lines',line=dict(color=VC,width=2),showlegend=False),1,2)
fig1.add_trace(go.Scatter(x=[],y=[],mode='lines',line=dict(color=NC,width=2),showlegend=False),1,3)
fig1.update_xaxes(title_text='step')
fig1.update_yaxes(type='log', title_text='cross-entropy', row=1, col=1)
fig1.update_yaxes(title_text='accuracy', range=[-0.02,1.02], row=1, col=2)
fig1.update_yaxes(title_text='||Θ||', row=1, col=3)
fig1.update_layout(template='plotly_white', height=320, width=1100, margin=dict(l=60,r=20,t=50,b=50),
                   legend=dict(x=0.02,y=1.18,orientation='h'),
                   title=f'Training dynamics -- ({task_str}) mod {P}, causal={CAUSAL}')
display(fig1)


# Figure 2: embedding diagnostics (PCA, Fourier spectrum)
fig2 = go.FigureWidget(make_subplots(rows=1, cols=2,
    subplot_titles=('Input embeddings (PC1 vs PC2)', 'Embedding Fourier magnitudes'),
    horizontal_spacing=0.15))
fig2.add_trace(go.Scatter(x=[0]*P, y=[0]*P, mode='markers+text',
    marker=dict(color=list(range(P)), colorscale='HSV', size=11, showscale=False),
    text=[str(i) for i in range(P)], textposition='top center', textfont=dict(size=8),
    showlegend=False, hovertemplate='k=%{text}<extra></extra>'), 1, 1)
fig2.add_trace(go.Bar(x=list(range(P//2+1)), y=[0]*(P//2+1), marker=dict(color='#2ca02c'), showlegend=False), 1, 2)
fig2.update_xaxes(title_text='PC1', row=1, col=1); fig2.update_yaxes(title_text='PC2', scaleanchor='x', scaleratio=1, row=1, col=1)
fig2.update_xaxes(title_text='frequency w', dtick=2, row=1, col=2); fig2.update_yaxes(title_text='|FFT|', row=1, col=2)
fig2.update_layout(template='plotly_white', height=420, width=1100, margin=dict(l=60,r=20,t=50,b=50), title='Embedding structure')
display(fig2)



### TRAINING LOOP ###
history = {k: [] for k in ['step','train_loss','val_loss','train_acc','val_acc','weight_norm']}
checkpoints = {}
snapshot = lambda: {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

# !!! NOTE !!!
# We use for the main training loop full batch GD but using sampling with replacement, that is, 
# we draw exactly as many samples per iteration as there are training instances. The reason behind this 
# is mainly related to training stability by giving the slingshot mechanism a harder time.
n_total_examples = P ** N_INPUTS
BATCH_SIZE = int(n_total_examples * TRAIN_FRAC)

for step in range(N_STEPS + 1):
    model.train()

    # sample random minibatch directly from GPU tensors
    idx = torch.randint(0, train_x.size(0), (BATCH_SIZE,), device=DEVICE)
    xb, yb = train_x[idx], train_y[idx]

    # forward pass & loss computation
    logits = model(xb)
    loss = F.cross_entropy(logits, yb)

    # backpropagation
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step(); scheduler.step()

    # validation during training to assess generalisation
    if step % LOG_EVERY == 0:
        model.eval()
        with torch.no_grad():
            # forward pass on full training set
            train_logits = model(train_x)
            train_acc    = (train_logits.argmax(-1) == train_y).float().mean().item()
            train_loss   = F.cross_entropy(train_logits, train_y).item()
            
            vl = model(val_x)
            val_loss, val_acc = F.cross_entropy(vl, val_y).item(), (vl.argmax(-1) == val_y).float().mean().item()
            
            wn = sum(p.pow(2).sum().item() for p in model.parameters()) ** 0.5

        for k, v in zip(history, [step, train_loss, val_loss, train_acc, val_acc, wn]):
            history[k].append(v)
    
        # snapshot every CHECKPOINT_EVERY steps
        if step % CHECKPOINT_EVERY == 0:
            with torch.no_grad():
                ref_x = val_x[:min(512, val_x.size(0))]
                ref_y = val_y[:min(512, val_y.size(0))]
                _, ref_cache = model(ref_x, return_cache=True)

            ckpt_data = dict(
                step             = step, 
                # accuracy values at checkpoint step, computed on full train & validation splits
                train_acc        = train_acc,
                val_acc          = val_acc,
                # snapshot of full training history up to this checkpoint
                history          = {k: list(v) for k, v in history.items()},
                model_state_dict = snapshot(),
                # store architecture and hyperparameters
                config           = dict(
                    p=P, n_inputs=N_INPUTS, train_frac=TRAIN_FRAC,
                    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF,
                    causal=CAUSAL, lr=LR, lr_min=LR_MIN, weight_decay=WEIGHT_DECAY,
                    betas=BETAS, n_steps=N_STEPS, seed=SEED,
                ),
                # store pre-computed activations from a forward pass on 512 val examples with return_cache=True
                cache            = {k: v.detach().cpu() for k, v in ref_cache.items()},
                ref_inputs       = ref_x.cpu(),
                ref_labels       = ref_y.cpu(),
                train_idx        = train_idx.cpu(),
                val_idx          = val_idx.cpu(),
            )
            label     = f'step_{step:06d}'
            checkpoints[label] = ckpt_data
            ckpt_path = os.path.join(CKPT_DIR, f'grok_ckpt_step{step}.pt')
            torch.save(ckpt_data, ckpt_path) 
            model.train() # restore model to training mode after intermediate model.eval()
            print(f'  >> saved {ckpt_path}  (train={train_acc:.3f}, val={val_acc:.3f})')

        pca_xy, fourier = embedding_diagnostics(model.token_emb.weight[:P])
        with fig1.batch_update():
            for i, k in enumerate(['train_loss','val_loss','train_acc','val_acc','weight_norm']):
                fig1.data[i].x = history['step']; fig1.data[i].y = history[k]
        with fig2.batch_update():
            fig2.data[0].x = pca_xy[:,0]; fig2.data[0].y = pca_xy[:,1]; fig2.data[1].y = fourier

# final checkpointing for completeness
final_path = os.path.join(CKPT_DIR, f'grok_ckpt_step{N_STEPS}.pt')
if not os.path.exists(final_path):
    model.eval()
    
    with torch.no_grad():
        ref_x = val_x[:min(512, val_x.size(0))]
        ref_y = val_y[:min(512, val_y.size(0))]
        _, ref_cache = model(ref_x, return_cache=True)
        
    ckpt_data = dict(
        step=N_STEPS, train_acc=train_acc, val_acc=val_acc,
        history={k: list(v) for k, v in history.items()},
        model_state_dict=snapshot(),
        config=dict(p=P, n_inputs=N_INPUTS, train_frac=TRAIN_FRAC,
                    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF,
                    causal=CAUSAL, lr=LR, lr_min=LR_MIN, weight_decay=WEIGHT_DECAY,
                    betas=BETAS, n_steps=N_STEPS, seed=SEED),
        cache={k: v.detach().cpu() for k, v in ref_cache.items()},
        ref_inputs=ref_x.cpu(), ref_labels=ref_y.cpu(),
        train_idx=train_idx.cpu(), val_idx=val_idx.cpu(),
    )
    torch.save(ckpt_data, final_path)
    print(f'  >> saved final checkpoint {final_path}')

print('Training complete.')

FigureWidget({
    'data': [{'line': {'color': '#1f77b4', 'width': 2},
              'mode': 'lines',
              'name': 'train',
              'type': 'scatter',
              'uid': 'c192668b-c54f-4799-a74e-afa60e3991d0',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'color': '#d62728', 'width': 2},
              'mode': 'lines',
              'name': 'val',
              'type': 'scatter',
              'uid': 'a07d7a34-56da-4ff0-84ac-2bedc745f4a5',
              'x': [],
              'xaxis': 'x',
              'y': [],
              'yaxis': 'y'},
             {'line': {'color': '#1f77b4', 'width': 2},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'uid': 'aba5d9aa-88b0-46f8-bb03-2e6698a9c739',
              'x': [],
              'xaxis': 'x2',
              'y': [],
              'yaxis': 'y2'},
             {'line': {'color': '#d6

FigureWidget({
    'data': [{'hovertemplate': 'k=%{text}<extra></extra>',
              'marker': {'color': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
                                   14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25,
                                   26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36],
                         'colorscale': [[0.0, '#ff0000'], [0.1111111111111111,
                                        '#ffa700'], [0.2222222222222222,
                                        '#afff00'], [0.3333333333333333,
                                        '#08ff00'], [0.4444444444444444,
                                        '#00ff9f'], [0.5555555555555556,
                                        '#00b7ff'], [0.6666666666666666,
                                        '#0010ff'], [0.7777777777777778,
                                        '#9700ff'], [0.8888888888888888,
                                        '#ff00bf'], [1.0, '#ff0000']],
                     

  >> saved ./ckpts_causal/grok_ckpt_step0.pt  (train=0.026, val=0.027)
  >> saved ./ckpts_causal/grok_ckpt_step1000.pt  (train=1.000, val=0.033)
  >> saved ./ckpts_causal/grok_ckpt_step2000.pt  (train=1.000, val=0.117)
  >> saved ./ckpts_causal/grok_ckpt_step3000.pt  (train=1.000, val=1.000)
  >> saved ./ckpts_causal/grok_ckpt_step4000.pt  (train=1.000, val=1.000)
  >> saved ./ckpts_causal/grok_ckpt_step5000.pt  (train=1.000, val=1.000)
  >> saved ./ckpts_causal/grok_ckpt_step6000.pt  (train=1.000, val=1.000)
  >> saved ./ckpts_causal/grok_ckpt_step7000.pt  (train=1.000, val=1.000)
  >> saved ./ckpts_causal/grok_ckpt_step8000.pt  (train=1.000, val=1.000)
  >> saved ./ckpts_causal/grok_ckpt_step9000.pt  (train=1.000, val=1.000)
  >> saved ./ckpts_causal/grok_ckpt_step10000.pt  (train=1.000, val=1.000)
  >> saved ./ckpts_causal/grok_ckpt_step11000.pt  (train=0.996, val=0.996)
  >> saved ./ckpts_causal/grok_ckpt_step12000.pt  (train=0.809, val=0.804)
  >> saved ./ckpts_causal/grok_ckpt_st